### Heart Disease Risk Prediction

In [1]:
%pip install pandas scikit-learn joblib xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [3]:
Data = pd.read_csv('heart_disease_risk_dataset_earlymed.csv')
Data.head()

# load the dataset

,Chest_Pain,Shortness_of_Breath,Fatigue,Palpitations,Dizziness,Swelling,Pain_Arms_Jaw_Back,Cold_Sweats_Nausea,High_BP,High_Cholesterol,Diabetes,Smoking,Obesity,Sedentary_Lifestyle,Family_History,Chronic_Stress,Gender,Age,Heart_Risk
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,48.0,0.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,46.0,0.0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,66.0,0.0
3,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,60.0,1.0
4,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,69.0,0.0


In [4]:
Data.isnull().sum().sum()

# check for missing values  

np.int64(0)

In [5]:
Data.duplicated().sum()

# check for duplicates values

np.int64(6245)

In [6]:
Data.drop_duplicates(inplace=True)

In [7]:
x = Data.drop(columns=['Heart_Risk'])
y = Data['Heart_Risk']

In [8]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# split the data

In [9]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(x_train)
X_test_scaled = scaler.transform(x_test)

In [10]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}


In [11]:
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred) * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred) * 100, 2),
        "F1-Score (%)": round(f1_score(y_test, y_pred) * 100, 2)
    })


In [12]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,   #step size while learning from each tree
    subsample=0.8,
    random_state=42   #controls randomness in splitting, to get consistent results
)

xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

# Append XGBoost to results
results.append({
    "Model": "XGBoost",
    "Accuracy (%)": round(accuracy_score(y_test, y_pred_xgb) * 100, 2),
    "Precision (%)": round(precision_score(y_test, y_pred_xgb) * 100, 2),
    "Recall (%)": round(recall_score(y_test, y_pred_xgb) * 100, 2),
    "F1-Score (%)": round(f1_score(y_test, y_pred_xgb) * 100, 2)
})

results_df = pd.DataFrame(results)
results_df.sort_values(by="Accuracy (%)", ascending=False)


,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%)
2,XGBoost,99.44,99.52,99.35,99.44
0,Logistic Regression,99.27,99.26,99.27,99.26
1,Decision Tree,98.21,98.41,97.98,98.19


In [15]:
import joblib

# Add XGBoost to models dict
models["XGBoost"] = xgb_model

# Pick best model automatically
best_model_name = results_df.sort_values(
    by="Accuracy (%)", ascending=False
).iloc[0]["Model"]

best_model = models[best_model_name]
print("Best Model Selected:", best_model_name)

# Save best model
joblib.dump(best_model, "../models/best_model.pkl")

# Save scaler (always needed)
joblib.dump(scaler, "../models/scaler.pkl")

# Save feature columns (critical for frontend)
feature_columns = x.columns.tolist()
joblib.dump(feature_columns, "../models/feature_columns.pkl")

Best Model Selected: XGBoost


['../models/feature_columns.pkl']